# Data Splitting M5 Forecasting Accuracy

Notebook ini mengimplementasikan **Tahap 3: Pembagian Data (Data Splitting)** sesuai `Outline Skripsi.docx`.

## Ringkasan metodologi Bab 3
- Data splitting dilakukan **sebelum preprocessing dan feature engineering** untuk mencegah data leakage.
- Main split bersifat kronologis: `d_1` sampai `d_1913` sebagai training set utama, dan `d_1914` sampai `d_1941` sebagai public test 28 hari.
- Evaluasi model menggunakan **4 fold time-series cross-validation** dengan skema **expanding window**.
- Semua statistik preprocessing pada tahap berikutnya harus di-fit hanya pada training split/fold masing-masing.

## Output notebook
Notebook ini menghasilkan artefak berikut pada folder `dataset/splits`:
- `calendar_split_labels.csv`
- `sales_train_main.csv`
- `sales_test_public.csv`
- `cv_folds.csv`
- `split_summary.json`


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import re

import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATASET_DIR = PROJECT_ROOT / "dataset"
SPLIT_DIR = DATASET_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

CALENDAR_PATH = DATASET_DIR / "calendar.csv"
SALES_PATH = DATASET_DIR / "sales_train_evaluation.csv"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATASET_DIR:   {DATASET_DIR}")
print(f"SPLIT_DIR:     {SPLIT_DIR}")


PROJECT_ROOT: C:\Users\adeli\speedrun-lulus
DATASET_DIR:   C:\Users\adeli\speedrun-lulus\dataset
SPLIT_DIR:     C:\Users\adeli\speedrun-lulus\dataset\splits


## 1. Load dataset mentah

Sesuai outline, tahap ini masih fokus pada pembagian data. Karena itu kita hanya memuat data yang diperlukan untuk menentukan batas waktu split:
- `calendar.csv` untuk pemetaan `d_x` ke tanggal dan minggu.
- `sales_train_evaluation.csv` untuk membentuk training utama (`d_1..d_1913`) dan public test (`d_1914..d_1941`).


In [3]:
calendar = pd.read_csv(CALENDAR_PATH)
sales = pd.read_csv(SALES_PATH)

print("calendar shape:", calendar.shape)
print("sales shape:   ", sales.shape)
calendar.head()


calendar shape: (1969, 14)
sales shape:    (30490, 1947)


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [4]:
id_columns = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_pattern = re.compile(r"^d_(\d+)$")
day_columns = [col for col in sales.columns if day_pattern.match(col)]

assert day_columns, "Tidak ditemukan kolom harian d_x pada sales_train_evaluation.csv"
assert "d_1941" in day_columns, "Kolom d_1941 wajib tersedia untuk public test 28 hari"

print("Jumlah kolom id: ", len(id_columns))
print("Jumlah kolom hari:", len(day_columns))
print("Kolom hari awal-akhir:", day_columns[0], "->", day_columns[-1])


Jumlah kolom id:  6
Jumlah kolom hari: 1941
Kolom hari awal-akhir: d_1 -> d_1941


## 2. Definisikan split waktu sesuai Bab 3

Berdasarkan outline skripsi:
- Training utama: `d_1 - d_1913`
- Public test: `d_1914 - d_1941`
- Cross-validation expanding window:
  - CV0: train `d_1 - d_1801`, valid `d_1802 - d_1829`
  - CV1: train `d_1 - d_1829`, valid `d_1830 - d_1857`
  - CV2: train `d_1 - d_1857`, valid `d_1858 - d_1885`
  - CV3: train `d_1 - d_1885`, valid `d_1886 - d_1913`


In [5]:
def day_num(day_label: str) -> int:
    match = day_pattern.match(day_label)
    if not match:
        raise ValueError(f"Format day label tidak valid: {day_label}")
    return int(match.group(1))


def make_day_range(start: str, end: str) -> list[str]:
    start_num = day_num(start)
    end_num = day_num(end)
    if end_num < start_num:
        raise ValueError(f"Range tidak valid: {start} -> {end}")
    return [f"d_{i}" for i in range(start_num, end_num + 1)]


main_split = {
    "training": {"name": "training", "start_day": "d_1", "end_day": "d_1913"},
    "test_public": {"name": "test_public", "start_day": "d_1914", "end_day": "d_1941"},
}

cv_specs = [
    {"fold": "CV0", "training_start_day": "d_1", "training_end_day": "d_1801", "validation_start_day": "d_1802", "validation_end_day": "d_1829"},
    {"fold": "CV1", "training_start_day": "d_1", "training_end_day": "d_1829", "validation_start_day": "d_1830", "validation_end_day": "d_1857"},
    {"fold": "CV2", "training_start_day": "d_1", "training_end_day": "d_1857", "validation_start_day": "d_1858", "validation_end_day": "d_1885"},
    {"fold": "CV3", "training_start_day": "d_1", "training_end_day": "d_1885", "validation_start_day": "d_1886", "validation_end_day": "d_1913"},
]

train_days = make_day_range(main_split["training"]["start_day"], main_split["training"]["end_day"])
test_days = make_day_range(main_split["test_public"]["start_day"], main_split["test_public"]["end_day"])

assert len(train_days) == 1913, "Training utama harus sepanjang 1913 hari"
assert len(test_days) == 28, "Public test harus sepanjang 28 hari"
assert set(train_days).isdisjoint(test_days), "Train dan test tidak boleh overlap"

print("Training days:", train_days[0], "->", train_days[-1], f"({len(train_days)} hari)")
print("Test days:    ", test_days[0], "->", test_days[-1], f"({len(test_days)} hari)")


Training days: d_1 -> d_1913 (1913 hari)
Test days:     d_1914 -> d_1941 (28 hari)


## 3. Bentuk artefak split utama

Tahap ini hanya memotong dataset berdasarkan kolom waktu, tanpa melakukan transformasi fitur apa pun. Ini konsisten dengan metodologi no-leakage pada Bab 3.


In [6]:
sales_train_main = sales[id_columns + train_days].copy()
sales_test_public = sales[id_columns + test_days].copy()

print("sales_train_main shape:", sales_train_main.shape)
print("sales_test_public shape:", sales_test_public.shape)
sales_test_public.head()


sales_train_main shape: (30490, 1919)
sales_test_public shape: (30490, 34)


,id,item_id,dept_id,cat_id,store_id,state_id,d_1914,d_1915,d_1916,d_1917,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,2,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,1,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,1,1,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,1,2,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,1,0,2,3,...,0,0,0,2,1,0,0,2,1,0


## 4. Label main split pada kalender

File ini berguna untuk tahap berikutnya karena memberi label setiap tanggal sebagai bagian dari `train` atau `test_public`.


In [7]:
calendar_split_labels = calendar.copy()
calendar_split_labels["main_split"] = pd.NA
calendar_split_labels.loc[calendar_split_labels["d"].isin(train_days), "main_split"] = "train"
calendar_split_labels.loc[calendar_split_labels["d"].isin(test_days), "main_split"] = "test_public"

assert calendar_split_labels["main_split"].notna().sum() == len(train_days) + len(test_days)
calendar_split_labels[calendar_split_labels["main_split"].notna()].tail()


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,main_split
1936,2016-05-18,11616,Wednesday,5,5,2016,d_1937,NaN,NaN,NaN,NaN,0,0,0,test_public
1937,2016-05-19,11616,Thursday,6,5,2016,d_1938,NaN,NaN,NaN,NaN,0,0,0,test_public
1938,2016-05-20,11616,Friday,7,5,2016,d_1939,NaN,NaN,NaN,NaN,0,0,0,test_public
1939,2016-05-21,11617,Saturday,1,5,2016,d_1940,NaN,NaN,NaN,NaN,0,0,0,test_public
1940,2016-05-22,11617,Sunday,2,5,2016,d_1941,NaN,NaN,NaN,NaN,0,0,0,test_public


## 5. Bentuk tabel cross-validation expanding window

Fold validasi disimpan sebagai metadata, sehingga pipeline modeling dan evaluasi dapat membaca batas waktu fold secara konsisten.


In [8]:
calendar_lookup = calendar.set_index("d")[["date", "wm_yr_wk"]]

cv_rows = []
for spec in cv_specs:
    train_range = make_day_range(spec["training_start_day"], spec["training_end_day"])
    valid_range = make_day_range(spec["validation_start_day"], spec["validation_end_day"])

    assert len(valid_range) == 28, f"Validation horizon {spec['fold']} harus 28 hari"
    assert set(train_range).isdisjoint(valid_range), f"Train/validation overlap pada {spec['fold']}"
    assert day_num(spec["training_end_day"]) + 1 == day_num(spec["validation_start_day"]), (
        f"{spec['fold']} harus kontigu antara training dan validation"
    )

    cv_rows.append(
        {
            "fold": spec["fold"],
            "training_start_day": spec["training_start_day"],
            "training_end_day": spec["training_end_day"],
            "validation_start_day": spec["validation_start_day"],
            "validation_end_day": spec["validation_end_day"],
            "training_days": len(train_range),
            "validation_days": len(valid_range),
            "training_start_date": calendar_lookup.loc[spec["training_start_day"], "date"],
            "training_end_date": calendar_lookup.loc[spec["training_end_day"], "date"],
            "validation_start_date": calendar_lookup.loc[spec["validation_start_day"], "date"],
            "validation_end_date": calendar_lookup.loc[spec["validation_end_day"], "date"],
        }
    )

cv_folds = pd.DataFrame(cv_rows)
cv_folds


,fold,training_start_day,training_end_day,validation_start_day,validation_end_day,training_days,validation_days,training_start_date,training_end_date,validation_start_date,validation_end_date
0,CV0,d_1,d_1801,d_1802,d_1829,1801,28,2011-01-29,2016-01-03,2016-01-04,2016-01-31
1,CV1,d_1,d_1829,d_1830,d_1857,1829,28,2011-01-29,2016-01-31,2016-02-01,2016-02-28
2,CV2,d_1,d_1857,d_1858,d_1885,1857,28,2011-01-29,2016-02-28,2016-02-29,2016-03-27
3,CV3,d_1,d_1885,d_1886,d_1913,1885,28,2011-01-29,2016-03-27,2016-03-28,2016-04-24


## 6. Bangun ringkasan split untuk dokumentasi pipeline

Ringkasan ini menyimpan informasi hari, tanggal, dan minggu awal-akhir agar pipeline berikutnya dapat mereferensi split secara eksplisit.


In [9]:
def build_period_summary(name: str, start_day: str, end_day: str) -> dict:
    day_range = make_day_range(start_day, end_day)
    return {
        "name": name,
        "start_day": start_day,
        "end_day": end_day,
        "n_days": len(day_range),
        "start_date": calendar_lookup.loc[start_day, "date"],
        "end_date": calendar_lookup.loc[end_day, "date"],
        "start_week": int(calendar_lookup.loc[start_day, "wm_yr_wk"]),
        "end_week": int(calendar_lookup.loc[end_day, "wm_yr_wk"]),
    }


split_summary = {
    "source": "Outline Skripsi Bab 3 - Tahap Pembagian Data (Data Splitting)",
    "main_split": {
        "training": build_period_summary("training", "d_1", "d_1913"),
        "test_public": build_period_summary("test_public", "d_1914", "d_1941"),
    },
    "cross_validation": [
        {
            "fold": row["fold"],
            "training": build_period_summary(
                f"{row['fold']}_training", row["training_start_day"], row["training_end_day"]
            ),
            "validation": build_period_summary(
                f"{row['fold']}_validation", row["validation_start_day"], row["validation_end_day"]
            ),
        }
        for _, row in cv_folds.iterrows()
    ],
    "artifacts": {
        "calendar_split_labels": str((SPLIT_DIR / "calendar_split_labels.csv").resolve()),
        "sales_train_main": str((SPLIT_DIR / "sales_train_main.csv").resolve()),
        "sales_test_public": str((SPLIT_DIR / "sales_test_public.csv").resolve()),
        "cv_folds": str((SPLIT_DIR / "cv_folds.csv").resolve()),
    },
    "notes": [
        "Split dilakukan sebelum feature engineering untuk mencegah data leakage.",
        "Main split mengikuti train d_1-d_1913 dan public test d_1914-d_1941.",
        "Empat fold expanding window disiapkan untuk time-series cross-validation.",
    ],
}

split_summary


{'source': 'Outline Skripsi Bab 3 - Tahap Pembagian Data (Data Splitting)',
 'main_split': {'training': {'name': 'training',
   'start_day': 'd_1',
   'end_day': 'd_1913',
   'n_days': 1913,
   'start_date': '2011-01-29',
   'end_date': '2016-04-24',
   'start_week': 11101,
   'end_week': 11613},
  'test_public': {'name': 'test_public',
   'start_day': 'd_1914',
   'end_day': 'd_1941',
   'n_days': 28,
   'start_date': '2016-04-25',
   'end_date': '2016-05-22',
   'start_week': 11613,
   'end_week': 11617}},
 'cross_validation': [{'fold': 'CV0',
   'training': {'name': 'CV0_training',
    'start_day': 'd_1',
    'end_day': 'd_1801',
    'n_days': 1801,
    'start_date': '2011-01-29',
    'end_date': '2016-01-03',
    'start_week': 11101,
    'end_week': 11549},
   'validation': {'name': 'CV0_validation',
    'start_day': 'd_1802',
    'end_day': 'd_1829',
    'n_days': 28,
    'start_date': '2016-01-04',
    'end_date': '2016-01-31',
    'start_week': 11549,
    'end_week': 11601}},
  

## 7. Simpan artefak split

Setelah sel ini dijalankan, tahap Data Splitting selesai dan artefak siap dipakai pada preprocessing Bab 3 berikutnya.


In [10]:
calendar_split_labels.to_csv(SPLIT_DIR / "calendar_split_labels.csv", index=False)
sales_train_main.to_csv(SPLIT_DIR / "sales_train_main.csv", index=False)
sales_test_public.to_csv(SPLIT_DIR / "sales_test_public.csv", index=False)
cv_folds.to_csv(SPLIT_DIR / "cv_folds.csv", index=False)

with open(SPLIT_DIR / "split_summary.json", "w", encoding="utf-8") as f:
    json.dump(split_summary, f, indent=2)

print("Artefak berhasil disimpan ke:")
for path in sorted(SPLIT_DIR.iterdir()):
    print("-", path.name)


Artefak berhasil disimpan ke:
- calendar_split_labels.csv
- cv_folds.csv
- sales_test_public.csv
- sales_train_main.csv
- split_summary.json


## 8. Sanity check akhir

Validasi ini membantu memastikan split sudah sesuai metodologi dan aman dari overlap waktu.


In [11]:
assert sales_train_main.columns[:6].tolist() == id_columns
assert sales_test_public.columns[:6].tolist() == id_columns
assert sales_train_main.columns[-1] == "d_1913"
assert sales_test_public.columns[6] == "d_1914"
assert sales_test_public.columns[-1] == "d_1941"
assert cv_folds["validation_days"].eq(28).all()

summary_df = pd.DataFrame(
    [
        {
            "split": "train_main",
            "n_series": sales_train_main.shape[0],
            "n_day_columns": sales_train_main.shape[1] - len(id_columns),
            "first_day": train_days[0],
            "last_day": train_days[-1],
        },
        {
            "split": "test_public",
            "n_series": sales_test_public.shape[0],
            "n_day_columns": sales_test_public.shape[1] - len(id_columns),
            "first_day": test_days[0],
            "last_day": test_days[-1],
        },
    ]
)

summary_df


,split,n_series,n_day_columns,first_day,last_day
0,train_main,30490,1913,d_1,d_1913
1,test_public,30490,28,d_1914,d_1941
